In [1]:
# fine-tune the bulkformer model on bioaid data
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  

import torch
from torch_geometric.typing import SparseTensor
from collections import OrderedDict
import polars as pl
import pandas as pd
import numpy as np

from tqdm.notebook import tqdm
import wandb

In [2]:
from utils.BulkFormer import BulkFormer
from model.config import model_params

### Load model

In [3]:
device = 'cuda'

In [4]:

graph_path = 'data/G_gtex.pt'
weights_path = 'data/G_gtex_weight.pt'
gene_emb_path = 'data/esm2_feature_concat.pt'

In [5]:
graph = torch.load(graph_path, map_location='cpu', weights_only=False)
weights = torch.load(weights_path, map_location='cpu', weights_only=False)
graph = SparseTensor(row=graph[1], col=graph[0], value=weights).t().to(device)
gene_emb = torch.load(gene_emb_path, map_location='cpu', weights_only=False)
model_params['graph'] = graph
model_params['gene_emb'] = gene_emb

In [6]:
torch.manual_seed(42)

In [7]:
model = BulkFormer(**model_params).to(device)

In [8]:
ckpt_model = torch.load('model/Bulkformer_ckpt_epoch_29.pt',weights_only=False)

In [9]:
new_state_dict = OrderedDict()
for key, value in ckpt_model.items():
    new_key = key[7:] if key.startswith("module.") else key
    new_state_dict[new_key] = value

model.load_state_dict(new_state_dict)

<All keys matched successfully>

### Load and preprocess fine-tune data

In [10]:
# first column is blank, not sure what this is?
df = pd.read_parquet("data/BioAID2_tpm_PC0.001_log2_genesymbol_dedup.parquet")
df

,5S_rRNA,A1BG,A1CF,A2M,A2ML1,A2MP1,A3GALT2,A4GALT,A4GNT,AAAS,...,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11AP1,ZYG11B,ZYX,ZYXP1,ZZEF1,ZZZ3
UID,,,,,,,,,,,,,,,,,,,,,
BIOAIDUR7635328,-9.965784,3.844421,-4.768566,-0.308443,-0.337883,-2.262744,0.140682,1.819860,-4.762749,3.214472,...,0.084193,1.056805,4.595255,-3.003912,-9.965784,2.883632,8.982770,-9.965784,4.520863,3.100828
BIOAIDUR7635329,-9.965784,2.714731,-4.924549,1.740522,-0.814740,-0.902754,0.900078,1.394203,-4.908236,2.361842,...,-1.446349,-0.478157,4.110858,-3.289820,-9.965784,2.195137,8.945683,-9.965784,3.639744,1.498775
BIOAIDUR7635330,-9.965784,3.094735,-5.190953,-0.636670,0.136676,-2.430200,-0.271796,2.572998,-9.965784,2.935585,...,-0.869367,0.353514,4.335027,-3.560427,-9.965784,2.405564,8.886435,-9.965784,4.217631,2.727056
BIOAIDUR7635331,-9.965784,3.858305,-4.201956,2.286416,-0.888058,-0.056439,-1.006053,2.782532,-3.616831,3.773280,...,1.422713,2.427880,4.809716,-2.899268,-9.965784,3.037024,8.194632,-9.965784,5.120731,4.067217
BIOAIDUR7635333,-9.965784,2.759780,-4.186961,2.198971,-0.201485,-2.139768,-1.568597,1.797288,-9.965784,1.772154,...,-1.102687,0.069633,3.677958,-5.044632,-9.965784,1.781691,8.924619,-9.965784,3.940051,2.661358
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
BIOAIDUR7638382,-9.965784,2.298589,-5.121351,0.957133,-0.387844,-1.919699,-1.469318,0.164746,-9.965784,1.959736,...,-1.743183,-0.304809,3.340619,-4.975002,-9.965784,1.660632,7.939103,-9.965784,3.089551,1.408697
BIOAIDUR7638383,-9.965784,0.687469,-4.760082,-0.986008,0.082238,-2.472122,-1.262456,-0.141490,-9.965784,2.464086,...,-0.561536,0.330481,3.659811,-5.005986,-9.965784,1.487138,9.018001,-9.965784,3.886367,1.776210
BIOAIDUR7638384,-9.965784,3.442350,-4.672972,-0.677458,-0.614155,-3.491373,-1.581673,1.480482,-5.110000,2.249983,...,-1.653321,-0.107698,3.475170,-3.069747,-9.965784,1.590637,9.198661,-9.965784,3.093366,1.351007


In [11]:
def main_gene_selection(X_df, gene_list):
    # fills columns (genes) that are in the gene list but not in our data
    # with -10
    # Returns df containing gene_list values (with -10 filling), columns that
    # were filled, and var, df indicating which columns are masked (to fill)

    to_fill_columns = list(set(gene_list) - set(X_df.columns))


    padding_df = pd.DataFrame(np.full((X_df.shape[0], len(to_fill_columns)), -10), 
                            columns=to_fill_columns, 
                            index=X_df.index)

    X_df = pd.DataFrame(np.concatenate([df.values for df in [X_df, padding_df]], axis=1), 
                        index=X_df.index, 
                        columns=list(X_df.columns) + list(padding_df.columns))
    X_df = X_df[gene_list]
    
    var = pd.DataFrame(index=X_df.columns)
    var['mask'] = [1 if i in to_fill_columns else 0 for i in list(var.index)]
    return X_df, to_fill_columns,var

In [12]:
bulkformer_gene_info = pd.read_csv('data/bulkformer_gene_info.csv')
bulkformer_gene_info = bulkformer_gene_info[bulkformer_gene_info['ensg_id'] != '35991']
bulkformer_gene_list = list(bulkformer_gene_info["gene_symbol"])

In [13]:
input_df , to_fill_columns, var= main_gene_selection(X_df=df,gene_list=bulkformer_gene_list)
input_df

,TSPAN6,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,NFYA,...,ENSG00000289763,ENSG00000289764,ENSG00000289766,ENSG00000289767,ENSG00000289768,ENSG00000289791,ENSG00000289809,ENSG00000290146,ENSG00000290147,ENSG00000290149
UID,,,,,,,,,,,,,,,,,,,,,
BIOAIDUR7635328,-3.596095,-9.965784,5.700050,2.922647,-10.0,10.055819,-0.038113,3.693386,2.843913,3.815707,...,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0
BIOAIDUR7635329,-5.949696,-9.965784,5.477522,2.584519,-10.0,9.970555,0.396137,3.283231,1.716843,2.801041,...,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0
BIOAIDUR7635330,-4.505073,-9.965784,5.299801,2.761906,-10.0,9.577174,1.573445,4.799567,3.419209,3.351787,...,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0
BIOAIDUR7635331,-2.106934,-9.965784,5.930045,3.491873,-10.0,9.186834,2.197080,4.866850,4.259811,4.396641,...,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0
BIOAIDUR7635333,-4.988783,-9.965784,5.191968,2.242854,-10.0,9.039704,-0.225765,3.175356,2.299190,3.305814,...,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
BIOAIDUR7638382,-2.818299,-9.965784,4.085649,1.670864,-10.0,7.816539,-0.964695,2.704331,1.753969,2.146238,...,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0
BIOAIDUR7638383,-5.817195,-9.965784,5.456692,3.832969,-10.0,9.882964,-0.075351,3.446624,2.236709,3.635924,...,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0
BIOAIDUR7638384,-3.708418,-9.965784,5.285920,3.132869,-10.0,9.287181,-1.055578,3.217550,1.689293,2.796272,...,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0,-10.0


In [14]:
"IFI27" in input_df and "MX1" in input_df

True

In [15]:
data = torch.tensor(input_df.values, dtype=torch.float32)
data

tensor([[ -3.5961,  -9.9658,   5.7001,  ..., -10.0000, -10.0000, -10.0000],
        [ -5.9497,  -9.9658,   5.4775,  ..., -10.0000, -10.0000, -10.0000],
        [ -4.5051,  -9.9658,   5.2998,  ..., -10.0000, -10.0000, -10.0000],
        ...,
        [ -3.7084,  -9.9658,   5.2859,  ..., -10.0000, -10.0000, -10.0000],
        [ -3.5800,  -4.3200,   5.1795,  ..., -10.0000, -10.0000, -10.0000],
        [ -3.2919,  -9.9658,   5.5064,  ..., -10.0000, -10.0000, -10.0000]])

In [16]:
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [17]:
MASK_PROB = 0.15
MASK_VALUE = -10.0

# 2. Masking function
def mask_inputs(x, mask_prob=MASK_PROB):
    """
    x: (batch_size, 20010) tensor
    Returns:
        masked_x: input with some values masked (set to MASK_VALUE)
        labels: target values (same shape), with unmasked positions = MASK_VALUE (so we can ignore them in loss)
        mask: boolean mask of which positions were masked
    """
    # Do not mask already missing positions
    valid_mask = (x != MASK_VALUE)
    rand = torch.rand_like(x)
    mask = (rand < mask_prob) & valid_mask

    masked_x = x.clone()
    masked_x[mask] = MASK_VALUE

    labels = torch.full_like(x, MASK_VALUE)
    labels[mask] = x[mask]

    return masked_x, labels, mask

In [18]:
def train(model, dataloader, num_epochs=10, lr=1e-4, device="cuda", accumulation_steps=8, wandb_project="Thesis"):
    # 🟢 Initialize wandb
    wandb.init(project=wandb_project, config={
        "learning_rate": lr,
        "epochs": num_epochs,
        "accumulation_steps": accumulation_steps,
        "batch_size": dataloader.batch_size,
        "model": model.__class__.__name__,
    })

    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss(reduction="none")
    scaler = torch.cuda.amp.GradScaler()

    model.train()
    global_step = 0

    for epoch in tqdm(range(num_epochs), desc="Epochs", position=0):
        running_loss = 0.0
        step = 0

        pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch+1}", leave=False, position=1)

        for i, (batch,) in pbar:
            batch = batch.to(device)
            masked_x, labels, mask = mask_inputs(batch)
            masked_x = masked_x.to(device)
            labels = labels.to(device)
            mask = mask.to(device)

            with torch.cuda.amp.autocast():
                preds = model(masked_x)
                loss_matrix = loss_fn(preds, labels)
                masked_loss = loss_matrix[mask].mean() / accumulation_steps

            scaler.scale(masked_loss).backward()
            running_loss += masked_loss.item()

            if (i + 1) % accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

                avg_loss = running_loss
                wandb.log({"loss": avg_loss, "step": global_step})
                pbar.set_postfix({"loss": f"{avg_loss:.6f}"})

                running_loss = 0.0
                step += 1
                global_step += 1

            del batch, masked_x, labels, mask, preds, loss_matrix, masked_loss
            torch.cuda.empty_cache()

        # Final gradient step flush
        if (i + 1) % accumulation_steps != 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            wandb.log({"loss": running_loss, "step": global_step})
            global_step += 1

    wandb.finish()

In [19]:
# about 8 minutes per epoch with batch size 1
dataloader = DataLoader(TensorDataset(data), batch_size=1, shuffle=True)
train(model, dataloader, num_epochs=5)

wandb: Currently logged in as: andrewwusyd (andrewwusyd-ucl) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/tmp/ipykernel_904427/960384320.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epochs:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 1:   0%|          | 0/1113 [00:00<?, ?it/s]

/tmp/ipykernel_904427/960384320.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


KeyboardInterrupt: 

: 

In [ ]:
torch.save(model.state_dict(), "fine-tuned-bulkformer.pt")